# Linear Regression and Its Cousins

## Initializations

In [1]:
# Required packages
import pandas as pd       # For data handling
import numpy as np        # For numerical operations (optional but useful)
import matplotlib.pyplot as plt  # For plotting
import seaborn as sns     # For nicer statistical plots
import scienceplots
from scipy.stats import skew
from scipy.stats import ttest_rel
from scipy.stats import binomtest
import statsmodels.api as sm
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.preprocessing import PowerTransformer # For transformations 
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA # For Principal Component Analysis
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import GridSearchCV, KFold, RepeatedStratifiedKFold, StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import cross_validate


import sys
import os

# Absolute path to the repo root
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if repo_root not in sys.path:
    sys.path.append(repo_root)

from utils import check_transform_suitability, selective_transform, near_zero_var, find_correlation, plot_corr

# Set the plotting style
sns.set_theme(style="darkgrid")  # Set theme for seaborn plots

# Load data required for the analysis in this notebook
two_cd = pd.read_parquet("../../data/twoClassData.parquet")
german_credit = pd.read_parquet("../../data/GermanCredit.parquet")
chem_man_pro = pd.read_parquet("../../data/ChemicalManufacturingProcess.parquet")
sol_train_x = pd.read_parquet("../../data/solTrainX.parquet")
sol_train_x_trans = pd.read_parquet("../../data/solTrainXtrans.parquet")
sol_test_x_trans = pd.read_parquet("../../data/solTestXtrans.parquet")
sol_train_y = pd.read_parquet("../../data/solTrainY.parquet")
sol_test_y = pd.read_parquet("../../data/solTestY.parquet")

## Ordinary Linear Regression

### Using sklearn

In [ ]:
X_train = sol_train_x_trans
X_test = sol_test_x_trans
y = sol_train_y
y_test = sol_test_y

lr = LinearRegression(fit_intercept=True)
lr.fit(X_train, y)

y_pred_train = lr.predict(X_train)

rmse = np.sqrt(mean_squared_error(y, y_pred_train))
r2 = r2_score(y, y_pred_train)

print("rmse: ", rmse)
print("R2: ", r2)


rmse:  0.48133135958274453
R2:  0.9446316255606921


In [4]:
# Evaluate on test set
y_pred_test = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

print("rmse: ", rmse)
print("R2: ", r2)

rmse:  0.7455801896508956
R2:  0.8709146842068273


### Using statsmodels for additional statistics

In [ ]:
X_train2 = sm.add_constant(X_train)

lr2 = sm.OLS(y, X_train2).fit()

print(lr2.summary())

                            OLS Regression Results                            
Dep. Variable:              solTrainY   R-squared:                       0.945
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     54.03
Date:                Thu, 18 Dec 2025   Prob (F-statistic):               0.00
Time:                        11:23:48   Log-Likelihood:                -654.04
No. Observations:                 951   AIC:                             1766.
Df Residuals:                     722   BIC:                             2878.
Df Model:                         228                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 2.4307      2.16

In [11]:
# Evaluate on test set
X_test2 = sm.add_constant(X_test)

y_pred_test_stats = lr2.predict(X_test2)

rmse_stats = np.sqrt(mean_squared_error(y_test, y_pred_test_stats))
r2_stats = r2_score(y_test, y_pred_test_stats)

print("rmse: ", rmse_stats)
print("R2: ", r2_stats)


rmse:  0.745580189650899
R2:  0.870914684206826


### Robust linear regression with Huber

In [15]:
from sklearn.linear_model import HuberRegressor

huber = HuberRegressor(
    epsilon=1.35,   # default Huber threshold (≈ MASS default)
    fit_intercept=True,
    max_iter=10000
)

huber.fit(X_train, y)

y_pred_huber = huber.predict(X_test)

rmse_huber = np.sqrt(mean_squared_error(y_test, y_pred_huber))
r2_huber = r2_score(y_test, y_pred_huber)

print("rmse: ", rmse_huber)
print("R2: ", r2_huber)


/Users/kasperboss/miniconda3/envs/apm_projects_env/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


rmse:  0.7561323554397766
R2:  0.8672349492709831
